In [11]:
import os
from pathlib import Path
import numpy as np
import polars as pl
import torch
from sentence_transformers import SentenceTransformer

# Anchor execution to repo root if running from notebooks/
if Path.cwd().name == "notebooks":
    os.chdir("..")
print("Working directory:", os.getcwd())

# MPS hardware acceleration check for Apple Silicon
device = "mps" if torch.backends.mps.is_available() else "cpu"
print(f"PyTorch acceleration device: {device}")

/Users/fi/Documents/Projects/DTSC-2302-Final-Project/.venv/lib/python3.10/site-packages/sentence_transformers/cross_encoder/CrossEncoder.py:13: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from tqdm.autonotebook import tqdm, trange


Working directory: /Users/fi/Documents/Projects/DTSC-2302-Final-Project
PyTorch acceleration device: mps


In [6]:
raw_books_path = "data/raw/goodreads_books_mystery_thriller_crime.json.gz"

print("Loading raw book metadata with Polars...")
books_df = (
    pl.read_ndjson(raw_books_path)
    .select([
        pl.col("book_id").cast(pl.Utf8),
        pl.col("title").fill_null(""),
        pl.col("description").fill_null(""),
        pl.col("average_rating").cast(pl.Float32, strict=False),
        pl.col("ratings_count").cast(pl.Int64, strict=False).fill_null(0)
    ])
    .filter(
        (pl.col("title") != "") & 
        (pl.col("description") != "")
    )
)

# 2. Save processed books table
output_path = "data/processed/clean_books.parquet"
books_df.write_parquet(output_path)
print(f"Saved {len(books_df):,} books to {output_path}")

Loading raw book metadata with Polars...
Saved 198,488 books to data/processed/clean_books.parquet


In [12]:
# Move working directory to project root if executed from notebooks/
if Path.cwd().name == "notebooks":
    os.chdir("..")
print("Anchored working directory to:", os.getcwd())

# 1. Load book IDs passing your EDA threshold
books = pl.read_parquet("data/processed/clean_books.parquet")
n_books = len(books)
embedding_dim = 384  # Standard output dimension for all-MiniLM-L6-v2

# 2. Generate normalized unit-length random vectors
mock_vectors = np.random.randn(n_books, embedding_dim).astype(np.float32)
mock_vectors /= np.linalg.norm(mock_vectors, axis=1, keepdims=True)

# 3. Export to disk for Notebook 03 development
np.save("data/processed/mock_embeddings_384d.npy", mock_vectors)
print(f"Exported mock embedding matrix: {mock_vectors.shape}")

Anchored working directory to: /Users/fi/Documents/Projects/DTSC-2302-Final-Project
Exported mock embedding matrix: (198488, 384)


In [10]:
# 1. Identify books meeting the 40-review minimum threshold
valid_books = (
    reviews_df.group_by("book_id")
    .agg(pl.len().alias("review_count"))
    .filter(pl.col("review_count") >= 40)
)

# 2. Aggregate the top review snippets per book into a single text block
aggregated_reviews = (
    reviews_df.join(valid_books, on="book_id", how="inner")
    .group_by("book_id")
    .agg(
        # Take the first 10-15 substantive reviews to control token length
        pl.col("cleaned_text")
        .head(15)
        .str.join(delimiter=" \n ")
        .alias("aggregated_reviews")
    )
)

# 3. Join with metadata and construct the unified pseudo-document
pseudo_docs_df = (
    books_df.join(aggregated_reviews, on="book_id", how="inner")
    .with_columns(
        pl.concat_str(
            [
                pl.lit("Title: "), pl.col("title"), pl.lit("\n"),
                pl.lit("Description: "), pl.col("description"), pl.lit("\n"),
                pl.lit("Reader Perspectives: "), pl.col("aggregated_reviews")
            ]
        ).alias("pseudo_document")
    )
    .select(["book_id", "title", "pseudo_document"])
)

# 4. Save to disk
pseudo_docs_df.write_parquet("data/processed/pseudo_documents.parquet")
print(f"Generated {len(pseudo_docs_df):,} pseudo-documents successfully.")

Generated 5,775 pseudo-documents successfully.


In [13]:
# Apple Silicon GPU acceleration
device = "mps" if torch.backends.mps.is_available() else "cpu"
print(f"Generating embeddings using device: {device}")

model = SentenceTransformer("all-MiniLM-L6-v2", device=device)

# Encode all 5,775 pseudo-documents
texts = pseudo_docs_df["pseudo_document"].to_list()
embeddings = model.encode(
    texts,
    batch_size=64,
    show_progress_bar=True,
    normalize_embeddings=True
)

# Export for team consumption
np.save("data/processed/book_embeddings_5775x384.npy", embeddings)
print(f"Saved {embeddings.shape} embeddings successfully.")

Generating embeddings using device: mps


Batches: 100%|██████████| 91/91 [01:25<00:00,  1.06it/s]

Saved (5775, 384) embeddings successfully.
